#### Imports and Load Data

In [ ]:
import os
import sys
import time
import libpysal
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from numba import njit, prange
from esda.moran import Moran
from scipy.stats import chi2_contingency

warnings.filterwarnings("ignore", category=UserWarning)
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import config
config.apply_plot_style()

In [ ]:
tracts = config.load_tracts(city_only=True)
geoids = tracts["GEOID"].tolist()
df_sp = config.load_theft_data(coords_required=True)
df_sp = df_sp[df_sp["GEOID"].isin(set(geoids))].copy()
gdf = gpd.GeoDataFrame(df_sp, geometry=gpd.points_from_xy(df_sp["lon_final"], df_sp["lat_final"]),
                       crs=config.LATLON_CRS).to_crs(config.SPATIAL_CRS)
counts = (df_sp.groupby(["GEOID", "year"]).size().unstack(fill_value=0)
          .reindex(index=geoids, columns=config.STUDY_YEARS, fill_value=0))
veh = config.annual_denominator("vehicles", geoids)
print(f"City tracts: {len(tracts):,}; theft points in tracts: {len(gdf):,}")

w_queen = libpysal.weights.Queen.from_dataframe(tracts, use_index=False)
w_queen.transform = "r"

#### Annual global Moran's I (city tracts)

With 999 permutations the smallest attainable pseudo p-value is 0.001.

In [ ]:
moran_rows = []
np.random.seed(42)
for y in config.STUDY_YEARS:
    mi = Moran(counts[y].values.astype(float), w_queen, permutations=999)
    moran_rows.append({"year": y, "moran_I": mi.I, "EI": mi.EI, "z_sim": mi.z_sim, "p_sim": mi.p_sim,
                       "geocoded_regime": y in config.GEOCODED_REGIME_YEARS})
moran_df = pd.DataFrame(moran_rows)
moran_df["q_BH"] = config.bh_fdr(moran_df["p_sim"])
moran_df.to_csv(os.path.join(config.TABLES_DIR, "table7_moran_annual.csv"), index=False)
print(moran_df.round(4).to_string(index=False))
# Trend test for a short, serially correlated annual series: Mann-Kendall with Hamed-Rao correction
stored_yrs = ~moran_df["geocoded_regime"]
mk_all = config.mann_kendall_hr(moran_df["moran_I"])
mk_st = config.mann_kendall_hr(moran_df.loc[stored_yrs, "moran_I"])
print(f"Trend in I (Mann-Kendall, Hamed-Rao): all years tau={mk_all['tau']:.2f}, Sen slope={mk_all['sen_slope']:.4f}/yr, "
      f"p={mk_all['p']:.3g}; stored-coordinate years tau={mk_st['tau']:.2f}, Sen slope={mk_st['sen_slope']:.4f}/yr, p={mk_st['p']:.3g}")
print(f"Mean I stored-regime years {moran_df.loc[stored_yrs, 'moran_I'].mean():.3f} vs geocoded-regime years "
      f"{moran_df.loc[~stored_yrs, 'moran_I'].mean():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_REGULAR)
config.shade_geocoded_regime(ax)
ax.plot(moran_df["year"], moran_df["moran_I"], color="#2166ac", lw=2.2, marker="o", zorder=4,
        label=f"Global Moran's I (all years p_sim <= {moran_df['p_sim'].max():.3f}); "
              f"Mann-Kendall trend tau = {mk_all['tau']:.2f}, p = {mk_all['p']:.2g}")
ax.set_xlabel("Year", fontsize=14)
ax.set_ylabel("Global Moran's I", fontsize=14)
ax.set_title("Spatial Clustering of Annual Theft Counts (queen, 999 permutations)\ngrey: years with geocoded locations", pad=12)
ax.set_xticks(config.STUDY_YEARS)
ax.set_xticklabels(config.STUDY_YEARS, rotation=45, ha="right")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig30_moran_annual.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

#### Knox test (exact distances, within-year permutation)

* Events: a stratified random sample of KNOX_SAMPLE_N thefts per year (fixed seed).
* Pairs: exact Euclidean distance <= d and |time gap| <= t (grid with cell = d is only used to find candidates).
* Null: occurrence dates are permuted **within each year**. Permuting across all 23 years would let year-to-year
  shifts in where thefts occur (including the switches between stored and geocoded coordinates) appear as
  space-time interaction.
* No cached results: the test is recomputed from the current data on every run.

In [ ]:
KNOX_DIST_M = [250, 500, 1000, 2000]
KNOX_TIME_DAYS = np.array([7, 30, 90], dtype=np.int64)
KNOX_PERMS = 999
KNOX_SAMPLE_N = 5000
SEED = 42

g = gdf.dropna(subset=["CRME_OCCUR_DT"]).copy()
g["t"] = (g["CRME_OCCUR_DT"] - pd.Timestamp("2000-01-01")).dt.days.astype(np.int64)
rng = np.random.default_rng(SEED)
parts = [grp.iloc[rng.choice(len(grp), size=min(KNOX_SAMPLE_N, len(grp)), replace=False)] for _, grp in g.groupby("year")]
ks = pd.concat(parts, ignore_index=True)
kx, ky, kt = ks.geometry.x.values, ks.geometry.y.values, ks["t"].values
year_groups = [np.flatnonzero(ks["year"].values == y) for y in sorted(ks["year"].unique())]
print(f"Knox sample: {len(ks):,} events ({KNOX_SAMPLE_N:,}/yr), "
      f"{(ks.duplicated(['lat_final', 'lon_final'])).mean()*100:.1f}% share a location with another sampled event")


@njit(parallel=True)
def _knox_counts(order, cstart, cend, nbr, xs, ys, ts, d2, T, d2_min=-1.0):
    n_cells = len(cstart)
    out = np.zeros((n_cells, len(T)), dtype=np.int64)
    for c in prange(n_cells):
        for a in range(cstart[c], cend[c]):
            i = order[a]
            for k in range(9):
                nc = nbr[c, k]
                if nc < 0:
                    continue
                for b in range(cstart[nc], cend[nc]):
                    j = order[b]
                    if j <= i:
                        continue
                    dx = xs[i] - xs[j]
                    dy = ys[i] - ys[j]
                    dd = dx * dx + dy * dy
                    if dd <= d2 and dd > d2_min:
                        dt = abs(ts[i] - ts[j])
                        for q in range(len(T)):
                            if dt <= T[q]:
                                out[c, q] += 1
    return out.sum(axis=0)


def build_cells(xs, ys, cell):
    ci = np.floor((xs - xs.min()) / cell).astype(np.int64)
    ri = np.floor((ys - ys.min()) / cell).astype(np.int64)
    key = ci * 1_000_003 + ri
    uniq, inv = np.unique(key, return_inverse=True)
    order = np.argsort(inv, kind="stable")
    cstart = np.searchsorted(inv[order], np.arange(len(uniq)))
    cend = np.append(cstart[1:], len(order))
    lookup = {k: i for i, k in enumerate(uniq)}
    nbr = np.full((len(uniq), 9), -1, dtype=np.int64)
    for i, k in enumerate(uniq):
        c, r = divmod(k, 1_000_003)
        m = 0
        for dc in (-1, 0, 1):
            for dr in (-1, 0, 1):
                nbr[i, m] = lookup.get((c + dc) * 1_000_003 + (r + dr), -1)
                m += 1
    return order, cstart, cend, nbr


def permute_within_years(ts, groups, rng):
    out = ts.copy()
    for idx in groups:
        out[idx] = ts[idx][rng.permutation(len(idx))]
    return out


knox_rows = []
for d_m in KNOX_DIST_M:
    d_ft = d_m * config.FT_PER_M
    order, cstart, cend, nbr = build_cells(kx, ky, d_ft)
    args = (order, cstart, cend, nbr, kx, ky)
    t0 = time.time()
    obs = _knox_counts(*args, kt, d_ft ** 2, KNOX_TIME_DAYS)
    rng_p = np.random.default_rng(SEED)
    perms = np.array([_knox_counts(*args, permute_within_years(kt, year_groups, rng_p), d_ft ** 2, KNOX_TIME_DAYS)
                      for _ in range(KNOX_PERMS)])
    for q, t_days in enumerate(KNOX_TIME_DAYS):
        pc = perms[:, q]
        knox_rows.append({"dist_m": d_m, "time_days": int(t_days), "observed": int(obs[q]), "perm_mean": pc.mean(),
                          "perm_sd": pc.std(), "ratio": obs[q] / pc.mean(),
                          "excess_pairs": obs[q] - pc.mean(),
                          "p_value": (np.sum(pc >= obs[q]) + 1) / (KNOX_PERMS + 1)})
    print(f"d={d_m:>5} m done in {time.time() - t0:.0f}s: " +
          ", ".join(f"t={r['time_days']}: ratio {r['ratio']:.3f}" for r in knox_rows[-3:]))
knox_df = pd.DataFrame(knox_rows)
knox_df["q_BH"] = config.bh_fdr(knox_df["p_value"])   # 12 distance x time bands
knox_df.to_csv(os.path.join(config.TABLES_DIR, "table8_knox_results.csv"), index=False)
print(knox_df.round(4).to_string(index=False))

Sensitivity: many events share exact coordinates (block-level or address points), so the shortest bands mix
near-repeats with same-location repeats. Re-run the 250 m and 500 m bands counting only pairs at distance > 0.

In [ ]:
knox_sens = []
for d_m in [250, 500]:
    d_ft = d_m * config.FT_PER_M
    order, cstart, cend, nbr = build_cells(kx, ky, d_ft)
    args = (order, cstart, cend, nbr, kx, ky)
    obs = _knox_counts(*args, kt, d_ft ** 2, KNOX_TIME_DAYS, 1e-6)
    rng_p = np.random.default_rng(SEED)
    perms = np.array([_knox_counts(*args, permute_within_years(kt, year_groups, rng_p), d_ft ** 2, KNOX_TIME_DAYS, 1e-6)
                      for _ in range(KNOX_PERMS)])
    for q, t_days in enumerate(KNOX_TIME_DAYS):
        pc = perms[:, q]
        knox_sens.append({"dist_m": d_m, "time_days": int(t_days), "pairs": "distance > 0 only", "observed": int(obs[q]),
                          "perm_mean": pc.mean(), "ratio": obs[q] / pc.mean(),
                          "p_value": (np.sum(pc >= obs[q]) + 1) / (KNOX_PERMS + 1)})
knox_sens = pd.DataFrame(knox_sens)
knox_sens = knox_sens.merge(knox_df[["dist_m", "time_days", "ratio"]].rename(columns={"ratio": "ratio_all_pairs"}),
                            on=["dist_m", "time_days"])
knox_sens.to_csv(os.path.join(config.TABLES_DIR, "table8b_knox_excluding_same_location.csv"), index=False)
print(knox_sens.round(4).to_string(index=False))

In [ ]:
kp = knox_df.pivot(index="dist_m", columns="time_days", values="ratio")
pp = knox_df.pivot(index="dist_m", columns="time_days", values="q_BH")
fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_REGULAR)
im = ax.imshow(kp.values, cmap="viridis", aspect="auto", vmin=1.0, vmax=max(kp.values.max(), 1.01))
ax.set_xticks(range(len(kp.columns))); ax.set_xticklabels([f"{t} days" for t in kp.columns])
ax.set_yticks(range(len(kp.index))); ax.set_yticklabels([f"{d} m" for d in kp.index])
for i, d in enumerate(kp.index):
    for j, t in enumerate(kp.columns):
        star = "*" if pp.loc[d, t] <= 0.05 else ""
        ax.text(j, i, f"{kp.loc[d, t]:.3f}{star}", ha="center", va="center", color="white", fontsize=14)
ax.set_xlabel("Time band", fontsize=14)
ax.set_ylabel("Distance band", fontsize=14)
ax.set_title(f"Knox Ratio (observed / within-year permuted mean)\n{KNOX_SAMPLE_N:,} thefts/yr, {KNOX_PERMS} permutations; * BH q <= 0.05", pad=12)
plt.colorbar(im, ax=ax, pad=0.02, shrink=0.8).set_label("Knox ratio")
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig31_knox_heatmap.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

#### Annual Gi* on theft rates: hotspot persistence

* Tracts: city tracts whose annual (interpolated) household-vehicle denominator is >= MIN_VEH in every year; the weights
  are built on this fixed set (tracts are excluded, not set to zero).
* Per year: Gi* on thefts per 1,000 household vehicles, queen weights, 9,999 permutations (two-sided p >= 0.0002,
  so Benjamini-Hochberg is not limited by the permutation resolution).
* A tract-year is a hotspot if HH and Benjamini-Hochberg q <= 0.05 within that year. The uncorrected p <= 0.05
  version is reported alongside for comparison.

In [ ]:
MIN_VEH = 100
# Valid tracts: a vehicle denominator in every year, >= MIN_VEH in every year, not a special land-use tract
valid = (veh.notna().all(axis=1) & (veh.min(axis=1) >= MIN_VEH)).values & ~config.is_special_use(geoids)
idx = np.flatnonzero(valid)
w_sub = libpysal.weights.w_subset(libpysal.weights.Queen.from_dataframe(tracts, use_index=False), idx)
rate = counts.values[idx] / veh.values[idx] * 1000
cnt_v = counts.values[idx].astype(float)
n_v, n_y = len(idx), len(config.STUDY_YEARS)
z_rate = np.full((n_v, n_y), np.nan); p_rate = np.full((n_v, n_y), np.nan); z_cnt = np.full((n_v, n_y), np.nan); p_cnt = np.full((n_v, n_y), np.nan)
for j, y in enumerate(config.STUDY_YEARS):
    z_rate[:, j], p_rate[:, j] = config.gi_star(rate[:, j], w_sub, permutations=9999)
    z_cnt[:, j], p_cnt[:, j] = config.gi_star(cnt_v[:, j], w_sub, permutations=9999)
q_rate = np.column_stack([config.bh_fdr(p_rate[:, j]) for j in range(n_y)])
hh_fdr = (z_rate > 0) & (q_rate <= 0.05)
hh_raw = (z_rate > 0) & (p_rate <= 0.05)
q_cnt = np.column_stack([config.bh_fdr(p_cnt[:, j]) for j in range(n_y)])
annual_hot = pd.DataFrame({"year": config.STUDY_YEARS,
                           "hotspots_rate_fdr": hh_fdr.sum(axis=0), "hotspots_rate_uncorrected": hh_raw.sum(axis=0),
                           "hotspots_count_fdr": ((z_cnt > 0) & (q_cnt <= 0.05)).sum(axis=0)})
annual_hot.to_csv(os.path.join(config.TABLES_DIR, "table9b_annual_hotspot_counts.csv"), index=False)
print(annual_hot.to_string(index=False))

tp = tracts.iloc[idx][["GEOID", "geometry"]].copy()
tp["persist_fdr"] = hh_fdr.sum(axis=1)
tp["persist_raw"] = hh_raw.sum(axis=1)
tp["mean_gi_z_rate"] = z_rate.mean(axis=1)
tp["std_gi_z_rate"] = z_rate.std(axis=1)
tp["mean_gi_z_count"] = z_cnt.mean(axis=1)
assert np.isfinite(z_rate).all(), "Gi* returned NaN: a tract has a missing rate in some year"
print(f"Persistence tracts: {n_v} of {len(tracts)} city tracts (vehicles >= {MIN_VEH} in every year, no special land use)")
print(f"Tract-years flagged HH: FDR {hh_fdr.sum():,}  vs uncorrected {hh_raw.sum():,}")

thr = [1, 5, 10, 15, 20, 23]
table9 = pd.DataFrame([{"threshold_years": t,
                        "n_tracts_fdr": int((tp["persist_fdr"] >= t).sum()),
                        "pct_fdr": round((tp["persist_fdr"] >= t).mean() * 100, 1),
                        "n_tracts_uncorrected": int((tp["persist_raw"] >= t).sum()),
                        "pct_uncorrected": round((tp["persist_raw"] >= t).mean() * 100, 1)} for t in thr])
table9.to_csv(os.path.join(config.TABLES_DIR, "table9_persistence_summary.csv"), index=False)
print(table9.to_string(index=False))

# Does the hotspot set change at the coordinate-regime switches? Fraction of hotspots retained year to year.
ret = pd.Series([(hh_fdr[:, j] & hh_fdr[:, j + 1]).sum() / max(hh_fdr[:, j].sum(), 1) for j in range(n_y - 1)],
                index=[f"{a}->{b}" for a, b in zip(config.STUDY_YEARS[:-1], config.STUDY_YEARS[1:])])
print("Share of year-t hotspots still hotspots in t+1:")
print(ret.round(3).to_string())

In [ ]:
PERSIST_BINS = [-0.5, 0.5, 4.5, 9.5, 14.5, 19.5, 22.5, 23.5]
PERSIST_LABELS = ["0 (never)", "1-4", "5-9", "10-14", "15-19", "20-22", "23 (all)"]
PCOL = [plt.cm.viridis(v) for v in np.linspace(0.05, 0.92, 7)]
fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_SQUARE)
tracts.iloc[np.flatnonzero(~valid)].plot(ax=ax, color="#aaaaaa", edgecolor="#888", linewidth=0.2)
tp.plot(ax=ax, column="persist_fdr", cmap=mcolors.ListedColormap(PCOL), norm=mcolors.BoundaryNorm(PERSIST_BINS, 7),
        edgecolor="#bbb", linewidth=0.15, zorder=3)
config.add_basemap(ax)
ax.set_axis_off()
ax.set_title("Years as Annual Gi* Hotspot (rate per 1,000 household vehicles, FDR q<=0.05)", pad=14)
handles = [mpatches.Patch(facecolor=PCOL[i], label=f"{PERSIST_LABELS[i]} (n={((tp['persist_fdr'] > PERSIST_BINS[i]) & (tp['persist_fdr'] < PERSIST_BINS[i+1])).sum()})")
           for i in range(7)]
handles.append(mpatches.Patch(facecolor="#aaaaaa", label=f"excluded: special land use or < {MIN_VEH} vehicles (n={(~valid).sum()})"))
ax.legend(handles=handles, title="Years as hotspot", loc="lower left", framealpha=0.92)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig32_persistence_rate.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_REGULAR)
sc = ax.scatter(tp["std_gi_z_rate"], tp["mean_gi_z_rate"], c=tp["persist_fdr"], cmap="viridis", vmin=0, vmax=23, s=12, alpha=0.7, lw=0)
plt.colorbar(sc, ax=ax, pad=0.02, shrink=0.8).set_label("Years as hotspot (rate-based, FDR)")
ax.set_xlabel("SD of annual Gi* z (theft rate) across 23 years")
ax.set_ylabel("Mean annual Gi* z (theft rate)")
ax.set_title("Hotspot Stability: Mean vs Variability of Annual Gi* z", pad=12)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig33_hotspot_stability_scatter.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

top20 = (tp.sort_values(["persist_fdr", "mean_gi_z_rate"], ascending=False).head(20)
         [["GEOID", "persist_fdr", "persist_raw", "mean_gi_z_rate", "std_gi_z_rate", "mean_gi_z_count"]])
top20.to_csv(os.path.join(config.TABLES_DIR, "table11_top20_persistent_rate.csv"), index=False)
print(top20.round(3).to_string(index=False))

#### Centre-of-mass drift

In [ ]:
com = df_sp.groupby("year").agg(com_lat=("lat_final", "mean"), com_lon=("lon_final", "mean"), n=("lat_final", "size")).reset_index()
com["geocoded_regime"] = com["year"].isin(config.GEOCODED_REGIME_YEARS)
st = com[~com["geocoded_regime"]]
res = {}
for col in ["com_lat", "com_lon"]:
    res[col] = (config.mann_kendall_hr(com[col]), config.mann_kendall_hr(st[col]))
    print(f"{col} (Mann-Kendall, Hamed-Rao): all years tau={res[col][0]['tau']:.3f} p={res[col][0]['p']:.3g} | "
          f"stored-coordinate years tau={res[col][1]['tau']:.3f} p={res[col][1]['p']:.3g}")
# Shift in km between regimes, to judge size
km_lat = 111.0
print(f"Mean centroid geocoded vs stored years: dlat = {(com.loc[com['geocoded_regime'], 'com_lat'].mean() - st['com_lat'].mean()) * km_lat:.2f} km")

fig, axes = plt.subplots(2, 1, figsize=config.FIGURE_SIZE_SQUARE, sharex=True)
for ax, col, lab, c in [(axes[0], "com_lat", "Centroid latitude (°N)", "#d6604d"), (axes[1], "com_lon", "Centroid longitude (°E)", "#2166ac")]:
    config.shade_geocoded_regime(ax)
    ax.plot(com["year"], com[col], color=c, lw=2.2, marker="o", zorder=3)
    r_all, r_st = res[col]
    ax.set_ylabel(lab, fontsize=14)
    ax.ticklabel_format(axis="y", useOffset=False, style="plain")
    ax.set_title(f"Mann-Kendall tau, all years {r_all['tau']:.2f} (p={r_all['p']:.2g}); "
                 f"stored-coordinate years {r_st['tau']:.2f} (p={r_st['p']:.2g})", fontsize=11)
    ax.spines[["top", "right"]].set_visible(False)
axes[1].set_xticks(config.STUDY_YEARS)
axes[1].set_xticklabels(config.STUDY_YEARS, rotation=45, ha="right")
fig.suptitle("Theft Centre of Mass by Year (grey: geocoded-location years)")
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig34_com_lat_lon_drift.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

#### Jaccard similarity of top-decile tracts

Top 10 % of all city tracts by annual count (zeros included, ties broken by GEOID so every set has the same size).
Year pairs that cross a coordinate-regime switch are flagged.

In [ ]:
k = int(np.ceil(0.10 * len(geoids)))
top_sets = {y: set(counts[y].sort_index().sort_values(ascending=False, kind="stable").index[:k]) for y in config.STUDY_YEARS}
Y = config.STUDY_YEARS
jac = np.array([[len(top_sets[a] & top_sets[b]) / len(top_sets[a] | top_sets[b]) for b in Y] for a in Y])
regime = {y: y in config.GEOCODED_REGIME_YEARS for y in Y}
lag1 = pd.DataFrame({"year_A": Y[:-1], "year_B": Y[1:], "jaccard": [jac[i, i + 1] for i in range(len(Y) - 1)]})
lag1["crosses_regime_switch"] = [regime[a] != regime[b] for a, b in zip(lag1["year_A"], lag1["year_B"])]
lag1.to_csv(os.path.join(config.TABLES_DIR, "table10_jaccard_lag1.csv"), index=False)
print(lag1.round(3).to_string(index=False))
print(f"Top set size: {k} tracts. Mean lag-1 Jaccard: within regime {lag1.loc[~lag1['crosses_regime_switch'], 'jaccard'].mean():.3f}, "
      f"across regime switch {lag1.loc[lag1['crosses_regime_switch'], 'jaccard'].mean():.3f}")
off = jac[~np.eye(len(Y), dtype=bool)]
print(f"Mean off-diagonal Jaccard: {off.mean():.3f}")

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(jac, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(Y))); ax.set_xticklabels(Y, rotation=90, fontsize=11)
ax.set_yticks(range(len(Y))); ax.set_yticklabels(Y, fontsize=11)
for i, y in enumerate(Y):
    if regime[y]:
        ax.get_xticklabels()[i].set_color("#b2182b"); ax.get_yticklabels()[i].set_color("#b2182b")
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02).set_label("Jaccard similarity")
ax.set_title(f"Jaccard Similarity of Top-10% Tracts (k={k})\nred labels: geocoded-location years", pad=14)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig35_jaccard_matrix.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

#### Diurnal pattern by persistence tier (known times only)

In [ ]:
def tier(p):
    return "Persistent (>=15 yr)" if p >= 15 else "Moderate (5-14 yr)" if p >= 5 else "Rare / never (<5 yr)"


TIERS = ["Persistent (>=15 yr)", "Moderate (5-14 yr)", "Rare / never (<5 yr)"]
TCOL = {"Persistent (>=15 yr)": "#b2182b", "Moderate (5-14 yr)": "#ef8a62", "Rare / never (<5 yr)": "#2980b9"}
tp["tier"] = tp["persist_fdr"].apply(tier)
jt = df_sp[~df_sp["time_unknown"]].dropna(subset=["hour"]).merge(tp[["GEOID", "tier"]], on="GEOID", how="inner")
cont = jt.groupby(["tier", "hour"]).size().unstack(fill_value=0).reindex(index=TIERS, columns=range(24), fill_value=0)
chi2, p_chi, dof, _ = chi2_contingency(cont.values)
cv = np.sqrt(chi2 / (cont.values.sum() * (min(cont.shape) - 1)))
print(f"χ² = {chi2:.0f}, df = {dof}, p = {p_chi:.3g}, Cramér's V = {cv:.3f}, n = {cont.values.sum():,}")
hp = cont.div(cont.sum(axis=1), axis=0) * 100
# Thefts from daytime parking are usually noticed and reported the same day
same_day = jt.assign(sd=jt["CRME_RPTD_DT"] == jt["CRME_OCCUR_DT"]).groupby("tier")["sd"].mean().reindex(TIERS) * 100
print("Share of thefts 07:00-08:59 (%):", hp[[7, 8]].sum(axis=1).round(1).to_dict())
print("Reported on the day of occurrence (%):", same_day.round(1).to_dict())

fig, (a1, a2) = plt.subplots(2, 1, figsize=config.FIGURE_SIZE_SQUARE, sharex=True, gridspec_kw={"height_ratios": [2, 1]})
for t in TIERS:
    a1.plot(range(24), hp.loc[t], color=TCOL[t], lw=2.2, marker="o", ms=4, label=f"{t} (n={cont.loc[t].sum():,})")
    if t != TIERS[-1]:
        a2.plot(range(24), hp.loc[t] - hp.loc[TIERS[-1]], color=TCOL[t], lw=1.6, label=f"{t} minus rare/never")
a2.axhline(0, color="#555", lw=0.9)
a1.set_ylabel("Share within tier (%)")
a1.set_title(f"Diurnal Theft Pattern by Hotspot Persistence (known times)\nχ² = {chi2:.0f}, Cramér's V = {cv:.3f}", pad=12)
a1.legend(frameon=False)
a2.set_ylabel("Difference (pp)")
a2.set_xlabel("Hour of day")
a2.set_xticks(range(0, 24, 2))
a2.set_xticklabels([f"{h:02d}:00" for h in range(0, 24, 2)], rotation=45, ha="right")
a2.legend(frameon=False, fontsize=9)
for a in (a1, a2):
    a.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig36_diurnal_persistence_tier.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()